# Sinhala Buddhist Corpus Builder - Document AI Version

This notebook extracts text from Sinhala Buddhist PDFs using Google Cloud Document AI.

**Key Features:**
- Direct PDF processing (no image conversion needed)
- Layout-aware text extraction
- Page-wise text file output (matching Vision API structure)
- Comprehensive logging and error handling
- Resume capability for interrupted processing

**Comparison with Vision API:**
- Vision API: PDF → Images → OCR
- Document AI: PDF → Direct extraction with layout preservation

## 1. Setup and Configuration

In [1]:
# Install required packages
!pip install --upgrade google-cloud-documentai google-auth google-cloud-storage tqdm -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.0/303.0 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.1/223.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 290.0/290.0 kB 6.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.43.0 which is incompatible.
google-adk 1.17.0 requires google-cloud-storage<3.0.0,>=2.18.0, but you have google-cloud-storage 3.5.0 which is incompatible.
google-cloud-aiplatform 1.122.0 requires google-cloud-storage<3.0.0,>=1.32.0; python_version < "3.13", but you have google-cloud-storage 3.5.0 which is incompatible.


In [2]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import time
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple
import re

from google.cloud import documentai_v1 as documentai
from google.oauth2 import service_account
from tqdm.notebook import tqdm
import pandas as pd

### Directory Configuration

Maintains the same structure as Vision API version for comparison.

In [14]:
# Base directory configuration
BASE_DIR = Path("/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/")
PDF_FOLDER = BASE_DIR / "data" / "raw" / "old_books"

# Document AI extraction folders (separate from Vision API)
EXTRACTION_BASE = BASE_DIR / "data" / "docai_extractions"
RAW_TEXT_DIR = EXTRACTION_BASE / "1_raw_text"
CLEANED_TEXT_DIR = EXTRACTION_BASE / "2_cleaned_text"

# Logs and metadata
LOGS_DIR = EXTRACTION_BASE / "logs"
PROCESSED_PDFS_DIR = BASE_DIR / "data" / "processed_pdfs"

# Create all directories
for directory in [RAW_TEXT_DIR, CLEANED_TEXT_DIR, LOGS_DIR, PROCESSED_PDFS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("✓ Directory structure created")
print(f"  PDFs from: {PDF_FOLDER}")
print(f"  Raw text to: {RAW_TEXT_DIR}")
print(f"  Cleaned text to: {CLEANED_TEXT_DIR}")
print(f"  Logs to: {LOGS_DIR}")

✓ Directory structure created
  PDFs from: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/old_books
  Raw text to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/1_raw_text
  Cleaned text to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/2_cleaned_text
  Logs to: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/docai_extractions/logs


### Google Cloud Authentication

**Setup Instructions:**
1. Place your service account JSON file in Google Drive
2. Update the path below to point to your credentials file
3. Ensure the service account has Document AI User role

In [5]:
# Path to your service account credentials JSON file
# TODO: Update this path to your actual credentials file location
CREDENTIALS_PATH = BASE_DIR / "auth" / "Vision OCR thematic runner.json"

# Alternative: If your credentials are in a different location, update here:
# CREDENTIALS_PATH = "/content/drive/MyDrive/path/to/your-service-account.json"

if not CREDENTIALS_PATH.exists():
    print("⚠️ WARNING: Credentials file not found!")
    print(f"   Looking for: {CREDENTIALS_PATH}")
    print("\n   Please update CREDENTIALS_PATH to point to your service account JSON file.")
else:
    print("✓ Credentials file found")

✓ Credentials file found


### Document AI Configuration

**Required Information:**
- Project ID: Your Google Cloud project ID
- Location: Processor location (usually 'us' or 'eu')
- Processor ID: Your Document AI OCR processor ID

**To find your processor ID:**
1. Go to Google Cloud Console → Document AI
2. Select your processor
3. Copy the Processor ID from the details page

In [7]:
# Document AI Configuration
# TODO: Update these with your actual values
PROJECT_ID = "thematic-runner-453210-s8"  # e.g., "buddhist-nlp-project"
LOCATION = "us"  # e.g., "us" or "eu"
PROCESSOR_ID = "11b17ff9bb09b5db"  # e.g., "abc123def456"

# Processing configuration
MAX_PAGES_PER_REQUEST = 15  # Document AI limit per synchronous request
RETRY_ATTEMPTS = 3
RETRY_DELAY = 2  # seconds

print("Configuration:")
print(f"  Project: {PROJECT_ID}")
print(f"  Location: {LOCATION}")
print(f"  Processor: {PROCESSOR_ID}")
print(f"  Max pages per request: {MAX_PAGES_PER_REQUEST}")

Configuration:
  Project: thematic-runner-453210-s8
  Location: us
  Processor: 11b17ff9bb09b5db
  Max pages per request: 15


## 2. Document AI Client Setup

In [8]:
def initialize_documentai_client(credentials_path: Path) -> documentai.DocumentProcessorServiceClient:
    """
    Initialize Document AI client with service account credentials.

    Args:
        credentials_path: Path to service account JSON file

    Returns:
        Initialized Document AI client
    """
    credentials = service_account.Credentials.from_service_account_file(
        str(credentials_path),
        scopes=["https://www.googleapis.com/auth/cloud-platform"]
    )

    client = documentai.DocumentProcessorServiceClient(credentials=credentials)
    return client

# Initialize client
try:
    docai_client = initialize_documentai_client(CREDENTIALS_PATH)
    print("✓ Document AI client initialized successfully")
except Exception as e:
    print(f"✗ Error initializing Document AI client: {e}")
    print("  Please check your credentials path and service account permissions.")

✓ Document AI client initialized successfully


## 3. Core Processing Functions

In [9]:
def get_processor_name(project_id: str, location: str, processor_id: str) -> str:
    """
    Construct the full processor resource name.

    Format: projects/{project}/locations/{location}/processors/{processor}
    """
    return f"projects/{project_id}/locations/{location}/processors/{processor_id}"


def process_pdf_with_documentai(
    client: documentai.DocumentProcessorServiceClient,
    pdf_path: Path,
    processor_name: str,
    max_retries: int = 3
) -> documentai.Document:
    """
    Process a PDF file using Document AI.

    Args:
        client: Document AI client
        pdf_path: Path to PDF file
        processor_name: Full processor resource name
        max_retries: Number of retry attempts for failed requests

    Returns:
        Processed document object

    Raises:
        Exception: If processing fails after all retries
    """
    # Read PDF file
    with open(pdf_path, "rb") as pdf_file:
        pdf_content = pdf_file.read()

    # Prepare request
    raw_document = documentai.RawDocument(
        content=pdf_content,
        mime_type="application/pdf"
    )

    request = documentai.ProcessRequest(
        name=processor_name,
        raw_document=raw_document
    )

    # Process with retries
    for attempt in range(max_retries):
        try:
            response = client.process_document(request=request)
            return response.document
        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = RETRY_DELAY * (2 ** attempt)  # Exponential backoff
                print(f"  Retry {attempt + 1}/{max_retries} after {wait_time}s: {str(e)[:100]}")
                time.sleep(wait_time)
            else:
                raise Exception(f"Failed after {max_retries} attempts: {e}")


def extract_text_by_page(document: documentai.Document) -> List[str]:
    """
    Extract text from Document AI response, organized by page.

    Args:
        document: Processed document from Document AI

    Returns:
        List of text strings, one per page
    """
    page_texts = []

    for page in document.pages:
        # Get text for this page using layout information
        page_text = ""

        # Extract text from paragraphs (maintains reading order)
        if hasattr(page, 'paragraphs') and page.paragraphs:
            for paragraph in page.paragraphs:
                # Get text segment for this paragraph
                text_segment = paragraph.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n"

        # Fallback: If no paragraphs, use lines
        elif hasattr(page, 'lines') and page.lines:
            for line in page.lines:
                text_segment = line.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + "\n"

        # Fallback: If no structured layout, use tokens
        elif hasattr(page, 'tokens') and page.tokens:
            for token in page.tokens:
                text_segment = token.layout.text_anchor.text_segments[0]
                start_index = text_segment.start_index if hasattr(text_segment, 'start_index') else 0
                end_index = text_segment.end_index
                page_text += document.text[start_index:end_index] + " "
            page_text = page_text.strip() + "\n"

        page_texts.append(page_text.strip())

    return page_texts


def save_page_texts(page_texts: List[str], output_dir: Path, pdf_name: str) -> None:
    """
    Save extracted text to individual page files.

    Creates files named: {pdf_name}/page_001.txt, page_002.txt, etc.
    Matches the Vision API output structure.

    Args:
        page_texts: List of text strings, one per page
        output_dir: Base output directory (RAW_TEXT_DIR or CLEANED_TEXT_DIR)
        pdf_name: Name of the PDF (without extension)
    """
    # Create subdirectory for this PDF
    pdf_output_dir = output_dir / pdf_name
    pdf_output_dir.mkdir(parents=True, exist_ok=True)

    # Save each page
    for page_num, page_text in enumerate(page_texts, start=1):
        page_filename = f"page_{page_num:03d}.txt"
        page_file_path = pdf_output_dir / page_filename

        with open(page_file_path, 'w', encoding='utf-8') as f:
            f.write(page_text)


print("✓ Core processing functions defined")

✓ Core processing functions defined


## 4. Text Cleaning Functions

Basic cleaning operations to improve text quality.

In [10]:
def clean_text(text: str) -> str:
    """
    Apply basic text cleaning operations.

    Operations:
    - Remove excessive whitespace
    - Normalize line breaks
    - Remove special characters (keeping Sinhala unicode)

    Args:
        text: Raw text to clean

    Returns:
        Cleaned text
    """
    if not text:
        return ""

    # Normalize whitespace
    text = re.sub(r'[ \t]+', ' ', text)  # Multiple spaces/tabs to single space
    text = re.sub(r'\n\s*\n', '\n\n', text)  # Multiple blank lines to double line break

    # Remove leading/trailing whitespace from each line
    lines = [line.strip() for line in text.split('\n')]
    text = '\n'.join(lines)

    # Remove completely empty lines (optional)
    text = re.sub(r'\n\n+', '\n\n', text)

    return text.strip()


def clean_and_save_texts(raw_text_dir: Path, cleaned_text_dir: Path, pdf_name: str) -> None:
    """
    Clean all page texts for a PDF and save to cleaned directory.

    Args:
        raw_text_dir: Directory containing raw text files
        cleaned_text_dir: Directory to save cleaned text files
        pdf_name: Name of the PDF (without extension)
    """
    raw_pdf_dir = raw_text_dir / pdf_name
    cleaned_pdf_dir = cleaned_text_dir / pdf_name
    cleaned_pdf_dir.mkdir(parents=True, exist_ok=True)

    # Process each page file
    for page_file in sorted(raw_pdf_dir.glob("page_*.txt")):
        with open(page_file, 'r', encoding='utf-8') as f:
            raw_text = f.read()

        cleaned_text = clean_text(raw_text)

        output_file = cleaned_pdf_dir / page_file.name
        with open(output_file, 'w', encoding='utf-8') as f:
            f.write(cleaned_text)


print("✓ Text cleaning functions defined")

✓ Text cleaning functions defined


## 5. Logging and Progress Tracking

In [11]:
class ProcessingLogger:
    """
    Tracks processing progress and logs results.
    """

    def __init__(self, logs_dir: Path):
        self.logs_dir = logs_dir
        self.timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.log_file = logs_dir / f"processing_log_{self.timestamp}.json"
        self.results = []

    def log_pdf_result(self, pdf_name: str, status: str, pages: int = 0,
                      error: str = None, processing_time: float = 0):
        """
        Log the result of processing a single PDF.
        """
        result = {
            "pdf_name": pdf_name,
            "status": status,
            "pages": pages,
            "processing_time_seconds": round(processing_time, 2),
            "timestamp": datetime.now().isoformat(),
            "error": error
        }
        self.results.append(result)

    def save_log(self):
        """
        Save all logged results to JSON file.
        """
        with open(self.log_file, 'w', encoding='utf-8') as f:
            json.dump({
                "session_start": self.timestamp,
                "total_pdfs": len(self.results),
                "successful": sum(1 for r in self.results if r["status"] == "success"),
                "failed": sum(1 for r in self.results if r["status"] == "failed"),
                "results": self.results
            }, f, indent=2, ensure_ascii=False)

    def get_summary(self) -> str:
        """
        Get a summary of processing results.
        """
        successful = sum(1 for r in self.results if r["status"] == "success")
        failed = sum(1 for r in self.results if r["status"] == "failed")
        total_pages = sum(r["pages"] for r in self.results if r["status"] == "success")
        total_time = sum(r["processing_time_seconds"] for r in self.results)

        summary = f"""
Processing Summary:
─────────────────────────────────
Total PDFs: {len(self.results)}
Successful: {successful}
Failed: {failed}
Total Pages: {total_pages}
Total Time: {total_time:.2f}s ({total_time/60:.2f} minutes)
Log saved to: {self.log_file.name}
"""
        return summary


def load_processed_pdfs(logs_dir: Path) -> set:
    """
    Load list of already processed PDFs from previous logs.
    Enables resuming interrupted processing.

    Args:
        logs_dir: Directory containing log files

    Returns:
        Set of successfully processed PDF names
    """
    processed = set()

    for log_file in logs_dir.glob("processing_log_*.json"):
        try:
            with open(log_file, 'r', encoding='utf-8') as f:
                log_data = json.load(f)
                for result in log_data.get("results", []):
                    if result["status"] == "success":
                        processed.add(result["pdf_name"])
        except Exception as e:
            print(f"Warning: Could not read log file {log_file.name}: {e}")

    return processed


print("✓ Logging functions defined")

✓ Logging functions defined


## 6. Main Processing Pipeline

In [16]:
def process_single_pdf(
    pdf_path: Path,
    client: documentai.DocumentProcessorServiceClient,
    processor_name: str,
    raw_text_dir: Path,
    cleaned_text_dir: Path,
    logger: ProcessingLogger
) -> Tuple[str, int]:
    """
    Process a single PDF through the complete pipeline.

    Pipeline:
    1. Process PDF with Document AI
    2. Extract text by page
    3. Save raw text files
    4. Clean and save cleaned text files
    5. Log results

    Args:
        pdf_path: Path to PDF file
        client: Document AI client
        processor_name: Full processor resource name
        raw_text_dir: Directory for raw text output
        cleaned_text_dir: Directory for cleaned text output
        logger: Processing logger instance

    Returns:
        Tuple of (status, page_count)
    """
    pdf_name = pdf_path.stem
    start_time = time.time()

    try:
        # Step 1: Process with Document AI
        document = process_pdf_with_documentai(
            client=client,
            pdf_path=pdf_path,
            processor_name=processor_name,
            max_retries=RETRY_ATTEMPTS
        )

        # Step 2: Extract text by page
        page_texts = extract_text_by_page(document)
        page_count = len(page_texts)

        # Step 3: Save raw text
        save_page_texts(page_texts, raw_text_dir, pdf_name)

        # Step 4: Clean and save cleaned text
        clean_and_save_texts(raw_text_dir, cleaned_text_dir, pdf_name)

        # Log success
        processing_time = time.time() - start_time
        logger.log_pdf_result(
            pdf_name=pdf_name,
            status="success",
            pages=page_count,
            processing_time=processing_time
        )

        return "success", page_count

    except Exception as e:
        # Log failure
        processing_time = time.time() - start_time
        error_msg = str(e)
        logger.log_pdf_result(
            pdf_name=pdf_name,
            status="failed",
            error=error_msg,
            processing_time=processing_time
        )

        return "failed", 0


def process_all_pdfs(
    pdf_folder: Path,
    client: documentai.DocumentProcessorServiceClient,
    processor_name: str,
    raw_text_dir: Path,
    cleaned_text_dir: Path,
    logs_dir: Path,
    skip_processed: bool = True
) -> ProcessingLogger:
    """
    Process all PDFs in the folder.

    Args:
        pdf_folder: Directory containing PDF files
        client: Document AI client
        processor_name: Full processor resource name
        raw_text_dir: Directory for raw text output
        cleaned_text_dir: Directory for cleaned text output
        logs_dir: Directory for log files
        skip_processed: If True, skip PDFs that were successfully processed before

    Returns:
        Processing logger with results
    """
    # Get list of PDFs
    pdf_files = sorted(pdf_folder.rglob("*.pdf"))

    if not pdf_files:
        print(f"⚠️ No PDF files found in {pdf_folder}")
        return None

    # Load previously processed PDFs
    processed_pdfs = load_processed_pdfs(logs_dir) if skip_processed else set()

    if processed_pdfs:
        print(f"ℹ️ Found {len(processed_pdfs)} previously processed PDFs (will skip)")

    # Filter out already processed
    pdfs_to_process = [
        pdf for pdf in pdf_files
        if pdf.stem not in processed_pdfs
    ]

    print(f"\n📚 Processing {len(pdfs_to_process)} PDFs (Total: {len(pdf_files)})\n")

    if not pdfs_to_process:
        print("✓ All PDFs already processed!")
        return None

    # Initialize logger
    logger = ProcessingLogger(logs_dir)

    # Process each PDF
    for pdf_path in tqdm(pdfs_to_process, desc="Processing PDFs"):
        print(f"\n📄 Processing: {pdf_path.name}")

        status, page_count = process_single_pdf(
            pdf_path=pdf_path,
            client=client,
            processor_name=processor_name,
            raw_text_dir=raw_text_dir,
            cleaned_text_dir=cleaned_text_dir,
            logger=logger
        )

        if status == "success":
            print(f"  ✓ Success: {page_count} pages extracted")
        else:
            print(f"  ✗ Failed (check logs for details)")

    # Save final log
    logger.save_log()

    return logger


print("✓ Main processing pipeline defined")

✓ Main processing pipeline defined


## 7. Run Extraction

**Before running:**
1. ✓ Mount Google Drive
2. ✓ Update credentials path
3. ✓ Update PROJECT_ID, LOCATION, PROCESSOR_ID
4. ✓ Verify PDF folder contains files

In [17]:
# Verify setup before processing
print("Pre-flight checks:")
print(f"  PDF folder exists: {PDF_FOLDER.exists()}")
print(f"  PDF count: {len(list(PDF_FOLDER.rglob('*.pdf')))}")
print(f"  Credentials exist: {CREDENTIALS_PATH.exists()}")
print(f"  Output directories ready: {RAW_TEXT_DIR.exists()}")
print(f"\nDocument AI config:")
print(f"  Project: {PROJECT_ID}")
print(f"  Processor: {PROCESSOR_ID}")

Pre-flight checks:
  PDF folder exists: True
  PDF count: 28
  Credentials exist: True
  Output directories ready: True

Document AI config:
  Project: thematic-runner-453210-s8
  Processor: 11b17ff9bb09b5db


In [18]:
# Build processor name
PROCESSOR_NAME = get_processor_name(PROJECT_ID, LOCATION, PROCESSOR_ID)
print(f"Full processor name: {PROCESSOR_NAME}")

Full processor name: projects/thematic-runner-453210-s8/locations/us/processors/11b17ff9bb09b5db


In [19]:
# Run the extraction pipeline
logger = process_all_pdfs(
    pdf_folder=PDF_FOLDER,
    client=docai_client,
    processor_name=PROCESSOR_NAME,
    raw_text_dir=RAW_TEXT_DIR,
    cleaned_text_dir=CLEANED_TEXT_DIR,
    logs_dir=LOGS_DIR,
    skip_processed=True  # Set to False to reprocess all PDFs
)


📚 Processing 28 PDFs (Total: 28)



Processing PDFs:   0%|          | 0/28 [00:00<?, ?it/s]


📄 Processing: බාගතකර_ගැනීම.pdf
  Retry 1/3 after 2s: 400 Request contains an invalid argument. [field_violations {
  field: "raw_document.content"
  desc
  Retry 2/3 after 4s: 400 Request contains an invalid argument. [field_violations {
  field: "raw_document.content"
  desc
  ✗ Failed (check logs for details)

📄 Processing: බාගතකර_ගැනීම.pdf
  Retry 1/3 after 2s: 400 Request contains an invalid argument. [field_violations {
  field: "raw_document.content"
  desc


KeyboardInterrupt: 

In [ ]:
# Display summary
if logger:
    print(logger.get_summary())
else:
    print("\n✓ No new PDFs to process or processing complete!")

## 8. Verification and Statistics

In [ ]:
def get_extraction_statistics(raw_text_dir: Path, cleaned_text_dir: Path) -> pd.DataFrame:
    """
    Generate statistics about extracted text.

    Returns:
        DataFrame with per-PDF statistics
    """
    stats = []

    for pdf_dir in sorted(raw_text_dir.iterdir()):
        if not pdf_dir.is_dir():
            continue

        pdf_name = pdf_dir.name
        page_files = list(pdf_dir.glob("page_*.txt"))

        # Calculate total characters
        total_chars = 0
        for page_file in page_files:
            with open(page_file, 'r', encoding='utf-8') as f:
                total_chars += len(f.read())

        stats.append({
            "PDF Name": pdf_name,
            "Pages": len(page_files),
            "Total Characters": total_chars,
            "Avg Chars/Page": total_chars // len(page_files) if page_files else 0
        })

    df = pd.DataFrame(stats)
    return df


# Generate and display statistics
stats_df = get_extraction_statistics(RAW_TEXT_DIR, CLEANED_TEXT_DIR)
print("\n" + "="*60)
print("EXTRACTION STATISTICS")
print("="*60)
print(stats_df.to_string(index=False))
print("\nTotals:")
print(f"  Total PDFs: {len(stats_df)}")
print(f"  Total Pages: {stats_df['Pages'].sum()}")
print(f"  Total Characters: {stats_df['Total Characters'].sum():,}")

In [ ]:
# Sample extracted text (first page of first PDF)
sample_pdfs = sorted(RAW_TEXT_DIR.iterdir())
if sample_pdfs:
    sample_pdf_dir = sample_pdfs[0]
    sample_page = list(sample_pdf_dir.glob("page_*.txt"))[0]

    print("\n" + "="*60)
    print(f"SAMPLE EXTRACTED TEXT")
    print(f"PDF: {sample_pdf_dir.name}")
    print(f"Page: {sample_page.name}")
    print("="*60)

    with open(sample_page, 'r', encoding='utf-8') as f:
        sample_text = f.read()

    # Display first 500 characters
    print(sample_text[:500])
    print("\n[...truncated...]" if len(sample_text) > 500 else "")

## 9. Comparison with Vision API (Optional)

If you've run both extraction methods, you can compare outputs here.

In [ ]:
def compare_extraction_methods(
    vision_raw_dir: Path,
    docai_raw_dir: Path,
    sample_pdf_name: str
) -> None:
    """
    Compare Vision API and Document AI outputs for a specific PDF.

    Args:
        vision_raw_dir: Directory with Vision API raw text
        docai_raw_dir: Directory with Document AI raw text
        sample_pdf_name: Name of PDF to compare (without extension)
    """
    vision_pdf_dir = vision_raw_dir / sample_pdf_name
    docai_pdf_dir = docai_raw_dir / sample_pdf_name

    if not vision_pdf_dir.exists():
        print(f"Vision API output not found for: {sample_pdf_name}")
        return

    if not docai_pdf_dir.exists():
        print(f"Document AI output not found for: {sample_pdf_name}")
        return

    # Compare first page
    vision_page = vision_pdf_dir / "page_001.txt"
    docai_page = docai_pdf_dir / "page_001.txt"

    with open(vision_page, 'r', encoding='utf-8') as f:
        vision_text = f.read()

    with open(docai_page, 'r', encoding='utf-8') as f:
        docai_text = f.read()

    print(f"\nComparison for: {sample_pdf_name} (Page 1)")
    print("="*60)
    print(f"Vision API length: {len(vision_text)} characters")
    print(f"Document AI length: {len(docai_text)} characters")
    print(f"Difference: {abs(len(vision_text) - len(docai_text))} characters")
    print("\nVision API (first 300 chars):")
    print("-"*60)
    print(vision_text[:300])
    print("\nDocument AI (first 300 chars):")
    print("-"*60)
    print(docai_text[:300])

# Example usage (update paths if you have Vision API outputs):
# VISION_RAW_DIR = BASE_DIR / "data" / "raw" / "sinhala_corpus" / "raw_text"
# if VISION_RAW_DIR.exists():
#     sample_pdf = list(RAW_TEXT_DIR.iterdir())[0].name
#     compare_extraction_methods(VISION_RAW_DIR, RAW_TEXT_DIR, sample_pdf)

## Summary and Next Steps

**What this notebook does:**
1. ✓ Processes PDFs using Google Cloud Document AI
2. ✓ Extracts text page-by-page (matching Vision API structure)
3. ✓ Saves both raw and cleaned versions
4. ✓ Provides detailed logging and statistics
5. ✓ Supports resuming interrupted processing

**Output structure:**
```
data/docai_extractions/
├── 1_raw_text/
│   ├── pdf_name_1/
│   │   ├── page_001.txt
│   │   ├── page_002.txt
│   │   └── ...
├── 2_cleaned_text/
│   ├── pdf_name_1/
│   │   ├── page_001.txt
│   │   └── ...
└── logs/
    └── processing_log_YYYYMMDD_HHMMSS.json
```

**Next steps:**
1. Compare Document AI vs Vision API outputs for quality
2. Decide which method produces better Sinhala text quality
3. Further text cleaning and normalization
4. Corpus preparation for pretraining/fine-tuning

**Advantages of Document AI:**
- Better layout understanding (preserves reading order)
- Direct PDF processing (no image conversion)
- Structured output with paragraph/line information
- Potentially more accurate for complex documents

**Considerations:**
- API costs per page
- Processing speed
- Actual quality on your Sinhala Buddhist texts (needs evaluation)